# 31. SAM Segment Anything 개념

이 노트북은 `30_Swin_Transformer_계층적_비전_모델.ipynb` 다음 단계로, Segment Anything Model(SAM)이 segmentation 문제를 어떻게 promptable segmentation으로 바꾸는지 이해합니다.

SAM은 특정 클래스 이름을 바로 예측하는 semantic segmentation 모델이라기보다, 사용자가 제공한 point, box, mask 같은 prompt를 바탕으로 해당 영역의 mask를 생성하는 foundation segmentation 모델입니다.

이번 노트북의 목표는 다음과 같습니다.

- SAM의 image encoder, prompt encoder, mask decoder 구조를 이해합니다.
- point prompt와 box prompt가 segmentation에 어떤 힌트를 주는지 확인합니다.
- class prediction과 promptable mask prediction의 차이를 구분합니다.
- SAM 결과를 사용할 때의 장점과 한계를 정리합니다.

## 31-1. 준비

실제 SAM 가중치를 다운로드하지 않고, 간단한 도형 이미지와 prompt를 사용해 promptable segmentation의 흐름을 시각화합니다.

In [ ]:
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image, ImageDraw

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break

plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(42)

## 31-2. SAM의 큰 구조

SAM은 크게 세 부분으로 생각할 수 있습니다.

```text
image
  -> image encoder
  -> image embedding

prompt(point, box, mask, text 등)
  -> prompt encoder
  -> prompt embedding

image embedding + prompt embedding
  -> mask decoder
  -> segmentation masks + scores
```

핵심은 이미지를 한 번 embedding해 두고, 다양한 prompt를 바꿔 넣으면서 여러 mask를 빠르게 얻을 수 있다는 점입니다.

In [ ]:
steps = ['Image\nEncoder', 'Image\nEmbedding', 'Mask\nDecoder', 'Mask']
prompt_steps = ['Prompt\nEncoder', 'Prompt\nEmbedding']

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.axis('off')
ax.set_title('SAM의 입력 흐름')

for i, label in enumerate(steps):
    x = i * 2.0
    ax.add_patch(Rectangle((x, 1.4), 1.25, 0.65, facecolor='#dbeafe', edgecolor='#2563eb', linewidth=2))
    ax.text(x + 0.625, 1.725, label, ha='center', va='center')
    if i < len(steps) - 1:
        ax.annotate('', xy=(x + 1.75, 1.725), xytext=(x + 1.25, 1.725), arrowprops=dict(arrowstyle='->'))

for i, label in enumerate(prompt_steps):
    x = i * 2.0
    ax.add_patch(Rectangle((x, 0.25), 1.25, 0.65, facecolor='#dcfce7', edgecolor='#16a34a', linewidth=2))
    ax.text(x + 0.625, 0.575, label, ha='center', va='center')
    if i < len(prompt_steps) - 1:
        ax.annotate('', xy=(x + 1.75, 0.575), xytext=(x + 1.25, 0.575), arrowprops=dict(arrowstyle='->'))

ax.annotate('', xy=(4.3, 1.35), xytext=(3.1, 0.9), arrowprops=dict(arrowstyle='->', color='#16a34a'))
ax.text(0.6, 2.25, 'image', ha='center', weight='bold')
ax.text(0.6, 0.05, 'prompt', ha='center', weight='bold')
ax.set_xlim(-0.3, 7.3)
ax.set_ylim(-0.1, 2.7)
plt.show()

## 31-3. 예제 이미지와 prompt

SAM의 prompt는 사용자가 어떤 객체를 원한다고 알려 주는 힌트입니다. 대표적인 prompt는 다음과 같습니다.

- positive point: 이 점이 포함된 객체를 찾아라.
- negative point: 이 점이 포함된 영역은 제외하라.
- box: 이 박스 안의 객체를 찾아라.
- mask: 기존 mask를 더 다듬어라.

In [ ]:
image = Image.new('RGB', (320, 220), color=(235, 240, 245))
draw = ImageDraw.Draw(image)
draw.ellipse((40, 45, 145, 160), fill=(235, 120, 75))
draw.rectangle((190, 60, 285, 155), fill=(70, 145, 215))
draw.ellipse((205, 75, 270, 140), fill=(90, 190, 120))

positive_point = (95, 105)
negative_point = (235, 105)
box_prompt = (30, 35, 155, 170)

fig, ax = plt.subplots(figsize=(7, 4))
ax.imshow(image)
ax.scatter(*positive_point, s=120, c='#22c55e', edgecolor='white', linewidth=2, label='positive point')
ax.scatter(*negative_point, s=120, c='#ef4444', edgecolor='white', linewidth=2, label='negative point')
x1, y1, x2, y2 = box_prompt
ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor='#f59e0b', linewidth=3, label='box prompt'))
ax.set_title('prompt 예시')
ax.axis('off')
ax.legend(loc='lower right')
plt.show()

## 31-4. Prompt로 mask 후보 만들기

실제 SAM은 image embedding과 prompt embedding을 mask decoder에 넣어 mask를 예측합니다. 여기서는 도형의 색상과 prompt 위치를 이용해 mask 후보를 단순화해서 만듭니다.

In [ ]:
arr = np.asarray(image).astype(np.float32)

def color_distance_mask(arr, point, threshold=95):
    x, y = point
    target_color = arr[y, x]
    distance = np.linalg.norm(arr - target_color, axis=2)
    return distance < threshold

point_mask = color_distance_mask(arr, positive_point, threshold=90)

x1, y1, x2, y2 = box_prompt
box_mask = np.zeros(point_mask.shape, dtype=bool)
box_mask[y1:y2, x1:x2] = True

negative_mask = color_distance_mask(arr, negative_point, threshold=80)
prompted_mask = point_mask & box_mask & ~negative_mask

print('point mask pixels:', int(point_mask.sum()))
print('box-constrained mask pixels:', int((point_mask & box_mask).sum()))
print('final prompted mask pixels:', int(prompted_mask.sum()))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))

axes[0].imshow(point_mask, cmap='gray')
axes[0].set_title('positive point 기반 후보')
axes[1].imshow(point_mask & box_mask, cmap='gray')
axes[1].set_title('box로 제한')
axes[2].imshow(prompted_mask, cmap='gray')
axes[2].set_title('negative point 제외')

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.show()

## 31-5. Mask overlay와 score

SAM은 하나의 prompt에 대해 여러 mask 후보와 score를 반환할 수 있습니다. ambiguity가 있는 prompt에서는 여러 가능한 mask를 보여 주고, 사용자가 가장 적절한 결과를 선택할 수 있습니다.

In [ ]:
overlay = arr.astype(np.uint8).copy()
overlay[prompted_mask] = (0.55 * overlay[prompted_mask] + 0.45 * np.array([255, 220, 40])).astype(np.uint8)

box_area = (x2 - x1) * (y2 - y1)
stability_score = prompted_mask.sum() / box_area

fig, ax = plt.subplots(figsize=(7, 4))
ax.imshow(overlay)
ax.scatter(*positive_point, s=120, c='#22c55e', edgecolor='white', linewidth=2)
ax.scatter(*negative_point, s=120, c='#ef4444', edgecolor='white', linewidth=2)
ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor='#f59e0b', linewidth=3))
ax.set_title(f'prompted mask overlay, score={stability_score:.2f}')
ax.axis('off')
plt.show()

## 31-6. SAM과 기존 segmentation 모델의 차이

| 관점 | U-Net/DeepLabV3/Mask R-CNN | SAM |
|---|---|---|
| 주요 입력 | image | image + prompt |
| 출력 의미 | 정해진 class 또는 instance mask | prompt가 가리키는 영역 mask |
| class label | 모델이 예측하는 경우가 많음 | 기본 SAM은 class 이름을 직접 붙이지 않음 |
| 사용 방식 | task-specific model | interactive/general segmentation tool |
| 강점 | 특정 데이터셋 task에 최적화 | 다양한 이미지와 prompt에 범용적으로 대응 |

SAM은 강력하지만 모든 문제를 자동으로 해결하지는 않습니다. class 이름이 필요한 경우에는 classifier, detector, text-image model 등 다른 모델과 함께 사용해야 할 수 있습니다.

## 정리

- SAM은 image encoder, prompt encoder, mask decoder로 구성된 promptable segmentation 모델입니다.
- point, box, mask 같은 prompt는 사용자가 원하는 객체 영역을 지정하는 힌트입니다.
- SAM의 출력은 class label보다 mask 후보와 score에 가깝습니다.
- 실제 응용에서는 detection, classification, annotation tool과 함께 사용하는 경우가 많습니다.

다음 노트북 `32_현대_비전_모델_정리.ipynb`에서는 지금까지 다룬 CNN, YOLO, segmentation 모델, Transformer 계열 모델의 위치를 한 번에 정리합니다.